# iprPy relax_dynamic calculation

In [1]:
# Standard library imports
import datetime

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-06-25 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('relax_dynamic')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# relax_dynamic calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

## Introduction

The relax_dynamic calculation style dynamically relaxes an atomic configuration for a specified number of timesteps.  Upon completion, the mean, $\langle X \rangle$, and standard deviation, $\sigma_X$, of all thermo properties, $X$, are computed for a specified range of times.  This method is meant to measure equilibrium properties of bulk materials, both at zero K and at various temperatures.

### Version notes

- 2018-07-09: Notebook added.
- 2019-07-30: Description updated and small changes due to iprPy version.
- v0.10.0: Version 0.10 update - potentials now loaded from database.
- 2020-09-22: Setup and parameter definition streamlined.
- v0.11.0: Notebook updated to reflect version 0.11.  Restart capability added in.
- v0.12.0: Method updated to support the LAMMPS library interface.
  
### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)
- The calculation reports the standard deviation, $\sigma_X$ of the measured properties not the standard error of the mean, $\sigma_{\langle X \rangle}$.  The two are related to each other according to $\sigma_{\langle X \rangle} = \sigma_X \sqrt{\frac{C}{N}}$, where $N$ is the number of samples taken of $X$, and $C$ is a statistical inefficiency due to the autocorrelation of the measurements with time.  Obtaining a proper estimate of $\sigma_{\langle X \rangle}$ requires either estimating $C$ from the raw thermo data (not done here), or only taking measurements sporadically to ensure the samples are independent.
- Good (low error) results requires running large simulations for a long time.  The reasons for this are:
  - Systems have to be large enough to avoid issues with fluctuations across the periodic boundaries.
  - Runs must first let the systems equilibrate before meaningful measurements can be taken.
  - The standard deviation, $\sigma$, of thermo properties is proportional to the number of atoms, $N_a$ as $\sigma \propto \frac{1}{\sqrt{N_a}}$.
  - The standard error, $\sigma_x$ of thermo properties is proportional to the number of samples taken, $N$ as $\sigma_x \propto \frac{1}{\sqrt{N}}$.


## Method and Theory

An initial system (and corresponding unit cell system) is supplied with box dimensions, $a_i^0$, close to the equilibrium values. A LAMMPS simulation then integrates the atomic positions and velocities for a specified number of timesteps.

The calculation script allows for the use of different integration methods:

- nve integrates atomic positions without changing box dimensions or the system's total energy.

- npt integrates atomic positions and applies Nose-Hoover style thermostat and barostat (equilibriate to specified T and P).

- nvt integrates atomic positions and applies Nose-Hoover style thermostat (equilibriate to specified T).

- nph integrates atomic positions and applies Nose-Hoover style barostat (equilibriate to specified P).

- nve+l integrates atomic positions and applies Langevin style thermostat (equilibriate to specified T).

- nph+l integrates atomic positions and applies Nose-Hoover style barostat and Langevin style thermostat (equilibriate to specified T and P).

__Notes__ on the different control schemes:

- The Nose-Hoover barostat works by rescaling the box dimensions according to the measured system pressures.

- The Nose-Hoover thermostat works by rescaling the atomic velocities according to the measured system temperature (kinetic energy). Cannot be used with a temperature of 0 K.

- The Langevin thermostat works by modifying the forces on all atoms with both a dampener and a random temperature dependent fluctuation. Used at 0 K, only the force dampener is applied.

__Notes__ on run parameter values. The proper time to reach equilibrium (equilsteps), and sample frequency to ensure uncorrelated measurements (thermosteps) is simulation dependent. They can be influenced by the potential, timestep size, crystal structure, integration method, presence of defects, etc. The default values of equilsteps = 20,000 and thermosteps = 100 are based on general rule-of-thumb estimates for bulk crystals and EAM potentials, and may or may not be adequate.


## 2. Display the underlying code

This section displays the underlying code used when calculation.calc() is called in Section 4. It is provided here allowing any users to see and understand how the calculation works.

Feel free to modify and test the functions for yourself!  You can do this by
1. Copy the Python code displayed here into a "Code" cell and run it.
2. In Section 2.2, set "savefiles = True" and run the cell to save any supporting non-python files to the working directory.
3. In Section 4, call the primary calculation function directly rather than using calculation.calc().

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "relax_dynamic.py"

# Python script created by Lucas Hale and Karina Stetsyuk

# Standard library imports
import datetime
from typing import Optional, Union

# http://www.numpy.org/
import numpy as np

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential, unitfloat
from atomman.lammps import LAMMPS, LAMMPSobj

def relax_dynamic(lammps_command: Union[str, LAMMPSobj],
                  system: am.System,
                  potential: lammpspotential,
                  mpi_command: Optional[str] = None,
                  pxx: unitfloat = 0.0,
                  pyy: unitfloat = 0.0,
                  pzz: unitfloat = 0.0,
                  pxy: unitfloat = 0.0,
                  pxz: unitfloat = 0.0,
                  pyz: unitfloat = 0.0,
                  temperature: float = 0.0,
                  integrator: Optional[str] = None,
                  equilsteps: int = 20000,
                  runsteps: int = 200000,
                  thermosteps: int = 100,
                  dumpsteps: Optional[int] = None,
                  restartsteps: Optional[int] = None,
                  createvelocities: bool = True,
                  randomseed: Optional[int] = None,
                  usefiles: bool = False) -> dict:
    """
    Performs a full dynamic relax on a given system at the given temperature
    to the specified pressure state.
    
    Parameters
    ----------
    lammps_command : str, LAMMPSEXE or LAMMPSLIB
        LAMMPS executable command, LAMMPS library name, or an atomman LAMMPS
        interface object.
    system : atomman.System
        The system to perform the calculation on.
    potential : PotentialLAMMPS or PotentialLAMMPSKIM
        The LAMMPS implemented potential to use.
    mpi_command : str, optional
        The MPI command for running LAMMPS in parallel.  If not given, LAMMPS
        will run serially.
    pxx : float or str, optional
        The value to relax the x tensile pressure component to (default is
        0.0).
    pyy : float or str, optional
        The value to relax the y tensile pressure component to (default is
        0.0).
    pzz : float or str, optional
        The value to relax the z tensile pressure component to (default is
        0.0).
    pxy : float or str, optional
        The value to relax the xy shear pressure component to (default is
        0.0).
    pxz : float or str, optional
        The value to relax the xz shear pressure component to (default is
        0.0).
    pyz : float or str, optional
        The value to relax the yz shear pressure component to (default is
        0.0).
    temperature : float, optional
        The temperature to relax at (default is 0.0).
    runsteps : int, optional
        The number of integration steps to perform (default is 220000).
    integrator : str or None, optional
        The integration method to use. Options are 'npt', 'nvt', 'nph',
        'nve', 'nve+l', 'nph+l'. The +l options use Langevin thermostat.
        (Default is None, which will use 'nph+l' for temperature == 0, and
        'npt' otherwise.)
    thermosteps : int, optional
        Thermo values will be reported every this many steps (default is
        100).
    dumpsteps : int or None, optional
        Dump files will be saved every this many steps (default is None,
        which sets dumpsteps equal to runsteps).
    restartsteps : int or None, optional
        Restart files will be saved every this many steps (default is None,
        which sets restartsteps equal to runsteps).
    equilsteps : int, optional
        The number of timesteps at the beginning of the simulation to
        exclude when computing average values (default is 20000).
    randomseed : int or None, optional
        Random number seed used by LAMMPS in creating velocities and with
        the Langevin thermostat.  (Default is None which will select a
        random int between 1 and 900000000.)
    
    Returns
    -------
    dict

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [5]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [6]:
lammps_command = 'lmp_serial'
mpi_command = None
#mpi_command = 'mpiexec -localonly 4'

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 3 Mar 2020


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [7]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. Initial unit cell system

- __ucell__ is an atomman.System representing a fundamental unit cell of the system (required).  Here, this is generated using the load parameters and symbols.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [8]:
# Create ucell by loading prototype record
ucell = am.load('prototype', 'A1--Cu--fcc', symbols='Ni', a=3.5)

print(ucell)

avect =  [ 3.500,  0.000,  0.000]
bvect =  [ 0.000,  3.500,  0.000]
cvect =  [ 0.000,  0.000,  3.500]
origin = [ 0.000,  0.000,  0.000]
natoms = 4
natypes = 1
symbols = ('Ni',)
pbc = [ True  True  True]
per-atom properties = ['atype', 'pos']
     id |   atype |  pos[0] |  pos[1] |  pos[2]
      0 |       1 |   0.000 |   0.000 |   0.000
      1 |       1 |   0.000 |   1.750 |   1.750
      2 |       1 |   1.750 |   0.000 |   1.750
      3 |       1 |   1.750 |   1.750 |   0.000


### 3.4. System modifications

- __sizemults__ list of three integers specifying how many times the ucell vectors of $a$, $b$ and $c$ are replicated in creating system.

- __system__ is an atomman.System to perform the scan on (required). 

In [9]:
sizemults = [10, 10, 10]

# Generate system by supersizing ucell
system = ucell.supersize(*sizemults)
print('# of atoms in system =', system.natoms)

# of atoms in system = 4000


### 3.5. Calculation-specific parameters

- __pressure_xx__ gives the xx component of the pressure to equilibriate the system to (npt, nph, and nph+l styles).
- __pressure_yy__ gives the yy component of the pressure to equilibriate the system to (npt, nph, and nph+l styles).
- __pressure_zz__ gives the zz component of the pressure to equilibriate the system to (npt, nph, and nph+l styles).
- __pressure_xy__ gives the xy component of the pressure to equilibriate the system to (npt, nph, and nph+l styles).
- __pressure_xz__ gives the xz component of the pressure to equilibriate the system to (npt, nph, and nph+l styles).
- __pressure_yz__ gives the yz component of the pressure to equilibriate the system to (npt, nph, and nph+l styles).
- __temperature__ gives the temperature to equilibriate the system to (nvt, npt, nve+l, and nph+l styles).
- __integrator__ specifies the integrator style to use. Default value is 'nph+l' for temperature = 0, and 'npt' otherwise.
- __runsteps__ is the total number of integration timesteps to perform. Default value is 220000.
- __thermosteps__ specifies to output thermo values every this many timesteps. Default value is 100.
- __dumpsteps__ specifies to output dump files every this many timesteps. Default value is runsteps (only first and last steps are outputted as dump files).
- __dumpsteps__ specifies to output restart files every this many timesteps. Default value is runsteps (only last step is outputted as a restart file).    
- __equilsteps__ is the number of timesteps to equilibriate the system for. Only thermo values associated with timesteps greater than equilsteps will be included in the mean and standard deviation calculations. Default value is 20000. 
- __randomseed__ specifies a random number seed used to generate the initial atomic velocities and the Langevin thermostat fluctuations. Default value generates a new random integer every time.

In [10]:
pressure_xx = uc.set_in_units(0.0, 'GPa')
pressure_yy = uc.set_in_units(0.0, 'GPa')
pressure_zz = uc.set_in_units(0.0, 'GPa')
pressure_xy = uc.set_in_units(0.0, 'GPa')
pressure_xz = uc.set_in_units(0.0, 'GPa')
pressure_yz = uc.set_in_units(0.0, 'GPa')
temperature = 300.0
integrator = 'npt'
runsteps = 220000
thermosteps = 100
dumpsteps = runsteps
equilsteps = 20000
randomseed = None

## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [11]:
# What is calculation.calc an alias of?
calculation.calc.__module__

'iprPy.calculation.relax_dynamic.relax_dynamic'

In [12]:
results_dict = calculation.calc(lammps_command, system, potential,
                                mpi_command = mpi_command,
                                pxx = pressure_xx,
                                pyy = pressure_yy,
                                pzz = pressure_zz,
                                pxy = pressure_xy,
                                pxz = pressure_xz,
                                pyz = pressure_yz,
                                temperature = temperature,
                                runsteps = runsteps,
                                integrator = integrator,
                                thermosteps = thermosteps,
                                dumpsteps = dumpsteps,
                                equilsteps = equilsteps,
                                randomseed = randomseed)
print(results_dict.keys())

dict_keys(['dumpfile_initial', 'symbols_initial', 'dumpfile_final', 'symbols_final', 'E_pot', 'E_pot_stderr', 'E_total', 'E_total_stderr', 'lx', 'lx_stderr', 'ly', 'ly_stderr', 'lz', 'lz_stderr', 'xy', 'xy_stderr', 'xz', 'xz_stderr', 'yz', 'yz_stderr', 'measured_pxx', 'measured_pxx_stderr', 'measured_pyy', 'measured_pyy_stderr', 'measured_pzz', 'measured_pzz_stderr', 'measured_pxy', 'measured_pxy_stderr', 'measured_pxz', 'measured_pxz_stderr', 'measured_pyz', 'measured_pyz_stderr', 'temp', 'temp_stderr'])


### 4.2. Report results

Values returned in the results_dict:

- **'dumpfile_initial'** (*str*) - The name of the initial dump file
  created.
- **'symbols_initial'** (*list*) - The symbols associated with the
  initial dump file.
- **'dumpfile_final'** (*str*) - The name of the final dump file
  created.
- **'symbols_final'** (*list*) - The symbols associated with the final
  dump file.
- **'E_pot'** (*float*) - The mean measured potential energy.
- **'measured_pxx'** (*float*) - The measured x tensile pressure of the
  relaxed system.
- **'measured_pyy'** (*float*) - The measured y tensile pressure of the
  relaxed system.
- **'measured_pzz'** (*float*) - The measured z tensile pressure of the
  relaxed system.
- **'measured_pxy'** (*float*) - The measured xy shear pressure of the
  relaxed system.
- **'measured_pxz'** (*float*) - The measured xz shear pressure of the
  relaxed system.
- **'measured_pyz'** (*float*) - The measured yz shear pressure of the
  relaxed system.
- **'temp'** (*float*) - The mean measured temperature.
- **'E_pot_stderr'** (*float*) - The standard error of the measured
  potential energy values.
- **'measured_pxx_stderr'** (*float*) - The standard error of the
  measured x tensile pressure of the relaxed system.
- **'measured_pyy_stderr'** (*float*) - The standard error of the
  measured y tensile pressure of the relaxed system.
- **'measured_pzz_stderr'** (*float*) - The standard error of the
  measured z tensile pressure of the relaxed system.
- **'measured_pxy_stderr'** (*float*) - The standard error of the
  measured xy shear pressure of the relaxed system.
- **'measured_pxz_stderr'** (*float*) - The standard error of the
  measured xz shear pressure of the relaxed system.
- **'measured_pyz_stderr'** (*float*) - The standard error of the
  measured yz shear pressure of the relaxed system.
- **'temp_stderr'** (*float*) - The standard error of the measured temperature values.

In [13]:
# Show initial and final dump files
print(results_dict['dumpfile_initial'])
print(results_dict['symbols_initial'])
print(results_dict['dumpfile_final'])
print(results_dict['symbols_final'])

0.dump
('Ni',)
220000.dump
('Ni',)


In [15]:
energy_unit = 'eV'

print('Per-atom potential energy and standard error:')
print('E_pot =', uc.get_in_units(results_dict['E_pot'], energy_unit),
      '+-', uc.get_in_units(results_dict['E_pot_stderr'], energy_unit), energy_unit)

Per-atom potential energy and standard error:
E_pot = -4.413044326602825 +- 1.0688062780477982e-05 eV


In [16]:
length_unit = 'angstrom'

print('Box lengths, tilts and standard errors:')
print('lx =', uc.get_in_units(results_dict['lx'] / sizemults[0], length_unit),
      '+-', uc.get_in_units(results_dict['lx_stderr'] / sizemults[0], length_unit), length_unit)
print('ly =', uc.get_in_units(results_dict['ly'] / sizemults[1], length_unit),
      '+-', uc.get_in_units(results_dict['ly_stderr'] / sizemults[1], length_unit), length_unit)
print('lz =', uc.get_in_units(results_dict['lz'] / sizemults[2], length_unit),
      '+-', uc.get_in_units(results_dict['lz_stderr'] / sizemults[2], length_unit), length_unit)
print('xy =', uc.get_in_units(results_dict['xy'] / sizemults[1], length_unit),
      '+-', uc.get_in_units(results_dict['xy_stderr'] / sizemults[1], length_unit), length_unit)
print('xz =', uc.get_in_units(results_dict['xz'] / sizemults[2], length_unit),
      '+-', uc.get_in_units(results_dict['xz_stderr'] / sizemults[2], length_unit), length_unit)
print('yz =', uc.get_in_units(results_dict['yz'] / sizemults[2], length_unit),
      '+-', uc.get_in_units(results_dict['yz_stderr'] / sizemults[2], length_unit), length_unit)

Box lengths, tilts and standard errors:
lx = 3.533002095677728 +- 5.52765198117049e-05 angstrom
ly = 3.53300527684337 +- 5.720074565394354e-05 angstrom
lz = 3.533032828583841 +- 6.201108345662781e-05 angstrom
xy = 2.3834502872352955e-06 +- 5.62230809683467e-05 angstrom
xz = 8.043981627064538e-07 +- 5.609386200754536e-05 angstrom
yz = 4.515061725588449e-05 +- 5.487577768449238e-05 angstrom


In [17]:
pressure_unit = 'GPa'

# Show the computed pressure tensor
print('Pxx =', uc.get_in_units(results_dict['measured_pxx'], pressure_unit),
      '+-', uc.get_in_units(results_dict['measured_pxx_stderr'], pressure_unit), pressure_unit)
print('Pyy =', uc.get_in_units(results_dict['measured_pyy'], pressure_unit), 
      '+-', uc.get_in_units(results_dict['measured_pyy_stderr'], pressure_unit), pressure_unit)
print('Pzz =', uc.get_in_units(results_dict['measured_pzz'], pressure_unit), 
      '+-', uc.get_in_units(results_dict['measured_pzz_stderr'], pressure_unit), pressure_unit)
print('Pyz =', uc.get_in_units(results_dict['measured_pyz'], pressure_unit), 
      '+-', uc.get_in_units(results_dict['measured_pyz_stderr'], pressure_unit), pressure_unit)
print('Pxz =', uc.get_in_units(results_dict['measured_pxz'], pressure_unit), 
      '+-', uc.get_in_units(results_dict['measured_pxz_stderr'], pressure_unit), pressure_unit)
print('Pxy =', uc.get_in_units(results_dict['measured_pxy'], pressure_unit), 
      '+-', uc.get_in_units(results_dict['measured_pxy_stderr'], pressure_unit), pressure_unit)

Pxx = 0.0007045633631258227 +- 0.004027588101863566 GPa
Pyy = -0.0002261402928091773 +- 0.004088263159696231 GPa
Pzz = 0.00040363840847750903 +- 0.004185064712195981 GPa
Pyz = -0.0001511607707651381 +- 0.002339592748630846 GPa
Pxz = -0.0006145521734764213 +- 0.002405133344555094 GPa
Pxy = -4.924767152775669e-05 +- 0.002375793732210104 GPa
